# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is filled with a small, evidence-backed contract for the starter search-intelligence slice.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row is one content-page snapshot from the starter dataset. Each row summarizes the most recent 90 days of search and engagement signals for one page. This is a cross-sectional slice, not a full daily panel: the time window is the 90-day aggregated signal window built into the dataset.

In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
]

path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError(
        'Could not find data/raw/content_refresh_anonymized.csv.\n'
        'Expected it under the repo root or one level above the notebook folder.'
    )

print('Loading dataset from:', path)
df = pd.read_csv(path)

print('Rows in the slice:', len(df))
print('Unique content_id rows:', df['content_id'].nunique())
print('Duplicate content_id rows:', int((df.groupby('content_id').size() > 1).sum()))
print('\nSample row fields:')
print(df[['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'engagement_rate', 'trend_direction']].head(3).to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\content_refresh_anonymized.csv'

## 2. Fields: feature / label / context / excluded

Features:
- `impressions_90d` — traffic volume known at decision time
- `ctr` — click-through rate known at decision time
- `avg_position` — search position known at decision time
- `days_since_last_update` — freshness signal known at decision time
- `engagement_rate` — engagement signal known at decision time

Label / proxy:
- `decline_proxy = (trend_direction == 'down')` — a page that is currently declining in visibility.

Context:
- `content_id`, `client_id` — identifiers for grouping and validation
- `content_type`, `main_intent`, `competition_level` — page metadata that helps characterize the page but is not the target.

Excluded:
- `trend_direction` is excluded as a feature because it is the proxy label we would predict. Using it directly would leak the target.

In [ ]:
feature_columns = ['content_id', 'client_id', 'content_type', 'main_intent', 'competition_level',
                   'impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'engagement_rate']
feature_frame = df[feature_columns].copy()
feature_frame['decline_proxy'] = (df['trend_direction'] == 'down').astype(int)

print('Feature frame preview:')
print(feature_frame.head(5).to_string(index=False))
print('\nFeature frame shape:', feature_frame.shape)

## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below verify the contract claims:
1. Grain: one row per content page snapshot.
2. Slice size and coverage: rows in the starter slice and how many have the core feature signals.
3. Availability: how many rows survive the `IS TRUE`-style availability check for the planned features.

In [ ]:
# Grain check: one row per content page snapshot
page_counts = df.groupby('content_id').size()
duplicate_rows = int((page_counts > 1).sum())
print('Duplicate content_id rows:', duplicate_rows)

# Slice size and coverage
print('\nTotal rows in starter slice:', len(df))
print('Rows with impressions_90d > 0:', int((df['impressions_90d'] > 0).sum()))
print('Rows with sessions_90d > 0:', int((df['sessions_90d'] > 0).sum()))
print('Rows with trend_direction present:', int(df['trend_direction'].notna().sum()))
print('Distinct trend_direction values:', df['trend_direction'].unique())

# Availability check: all core features known at decision time
availability_mask = (
    (df['impressions_90d'] > 0)
    & df['ctr'].notna()
    & df['avg_position'].notna()
    & df['days_since_last_update'].notna()
    & df['engagement_rate'].notna()
)
print('\nRows with all core features available:', int(availability_mask.sum()))
print('Availability rate:', round(availability_mask.mean() * 100, 1), '%')

# Deliberate leakage experiment: add label-derived feature and then remove it
leaky_feature = (df['trend_direction'] == 'down').astype(int)
leak_score = (leaky_feature == leaky_feature).mean()
print('\nLeaky feature perfect self-match rate:', round(leak_score * 100, 1), '%')

# Honest baseline after removing the leaked target
honest_rule = (df['impressions_90d'] >= 100).astype(int)
honest_accuracy = (honest_rule == (df['trend_direction'] == 'down').astype(int)).mean()
print('Honest rule accuracy vs decline proxy:', round(honest_accuracy * 100, 1), '%')

## 4. Data limits

This starter slice cannot tell us whether a refresh caused an improvement. It is observational and cross-sectional, so the best it can support is ranking pages for review, not proving causal impact.

Limitations:
- The data is a snapshot of aggregated 90-day signals, not a full daily panel.
- The dataset has only anonymized page and client IDs, so it cannot reconstruct real URLs or perform raw audit tracing.
- It cannot separate seasonality or consolidation effects without additional historical windows.

The leakage trap is explicit: a label-derived feature built from `trend_direction` appears perfect, but it must be removed to keep the contract honest.

In [ ]:
# Confirm the notebook has a working availability check
print('Re-running the availability check:')
print('Rows with all core features available:', int(availability_mask.sum()))
print('Availability rate:', round(availability_mask.mean() * 100, 1), '%')
print('\nDecline proxy distribution:')
print(df['trend_direction'].value_counts(normalize=True).rename('share'))

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.